In [9]:
import redis

r = redis.StrictRedis(host="localhost", port=6379, password="", decode_responses=True)

for key in r.scan_iter("*"):
    print( f'{key} | {r.type(key)}')


iperf:udp:1:ue0 | list
iperf:udp:0:ue0 | list


In [6]:
### REMOVE ALL DATA
for key in r.scan_iter("*"):
    r.delete(key)
    

In [ ]:
# CONVERT ALL THE TIME SERIES TO DICTIONARY (containing lists)
import json
res = dict()

# Get Traffic traces from Redis
for key in r.scan_iter("*"):
    if r.type(key) == 'TSDB-TYPE':
        ts = r.ts().range(key, 0, "+")
        # Read values converting lists of tuples --> lists
        time_stamp, val = map(list, zip(*ts))
        ksplit = key.split(":")
        # print(ksplit)
        # res[ksplit[0]] = dict()
        # res[ksplit[0]][f'{ksplit[1]}:{ksplit[2]}'] = dict()
        # res[ksplit[0]][f'{ksplit[1]}:{ksplit[2]}'][ksplit[3]] = dict()
        # res[ksplit[0]][f'{ksplit[1]}:{ksplit[2]}'][ksplit[3]]["time"] = [i/1e3 for i in time_stamp]
        # res[ksplit[0]][f'{ksplit[1]}:{ksplit[2]}'][ksplit[3]]["val"]  = val
        res[key] = dict()
        res[key][ksplit[3]] = dict()
        res[key][ksplit[3]]["time"] = [i/1e3 for i in time_stamp]
        res[key][ksplit[3]]["val"]  = val
print( json.dumps(res,indent=2) )


In [ ]:
# SAVE REDIS TIMESERIES TO FILE
import json, pickle

ts = dict()
for key in r.scan_iter("*"):
    if r.type(key) == 'TSDB-TYPE':
        ts[key] = r.ts().range(key, 0, "+")

print( f'type of ts: {type(ts)}' )
print( f'type of ts["mec_host:ogstun_recv"]: {type(ts["mec_host:ogstun_recv"])}' )


# To save create a dictionary with [key = variable name] = [value = variable]
to_save = dict()
to_save["ts"] = ts
to_save["a"] = 123
to_save["b"] = dict()
to_save["b"]["aaa"] = 4567

print(f'Variables to save:\n {to_save}')

# SAVE TO FILE
with open('tmp.pkl', 'wb') as f:  # Python 3: open(..., 'wb')
    pickle.dump(to_save, f)
    # pickle.dump(json.dump(ts), f)


In [ ]:
# LOAD FROM FILE
import pickle
with open('tmp.pkl', 'rb') as f:
    obj = pickle.load(f)
print("-------------------")
print(f'Variables loaded from file:\n {obj}')
print("-------------------")
# print(type(obj["ts"]['mec_host:ogstun_recv']))
print( f'type of ts: {type(obj["ts"])}' )
print( f'type of ts["mec_host:ogstun_recv"]: {type(obj["ts"]["mec_host:ogstun_recv"])}' )

# If I want original varables I can use 'obj' keys to create variables
for k in obj:
    locals()[k] = obj[k]
print(f'ts = {ts}')
print(f'a  = {a}')
print(f'b  = {b}')


In [ ]:
# Read values with pandas
# import pandas as pd
# import pyarrow as pa

# r = redis.Redis(host="localhost", port=6379, password="", decode_responses=True)

# data = r.get("traffic")

import pandas as pd
import redis
import zlib
import pickle

df = pd.DataFrame({'A':[1,2,3]})
# r = redis.Redis(host='localhost', port=6379, db=0)
r = redis.Redis(host="localhost", port=6379, password="", decode_responses=True)
# r.set("key", zlib.compress( pickle.dumps(df)))
df = pickle.loads(zlib.decompress(r.get("cpu")))


In [ ]:
### Push data to timeseries
import time, datetime, json

r.lpush('startsim', datetime.datetime.now().timestamp())

for i in range(10):
    # a = dict()
    # a["val1"] = i
    # a["val2"] = i/2
    # b = json.dumps(a)
    r.ts().add('test2', "*", i) # automatic timestamp in ms
    time.sleep(0.1)

print( "-------------------" )
print( r.ts().range("test2", 0,"+") )
# print( r.ts().get("test2") )


In [ ]:
### CHECK LISTS
r.lpush('mec_host:LanguageList', 1.1)
r.lpush('mec_host:LanguageList', 3.1)
r.lpush('mec_host:LanguageList', 4.2)

for key in r.scan_iter("*"):
    print(key)
print( "-------------------" )
print( r.lrange( "mec_host:LanguageList", 0, -1 ) )


In [ ]:
# PUSH a list
import redis
rc = redis.StrictRedis(host="localhost", port=6379, password="", decode_responses=True)

# Add values to the Redis list through the HEAD position of the list
rc.lpush('alist', 1.2)
# rc.lpush('alist', 1.3)
# rc.lpush('alist', (1,2))
# rc.lpush('alist', (3,4))

# Push multiple values through the HEAD of the list
# rc.lpush('alist', 1.1)

for key in r.scan_iter("*"):
    print( f'{key} | {r.type(key)}')

print( "-------------------" )
print( rc.lrange( "alist", 0, -1 ) )
print( rc.llen('alist') )
print( "-------------------" )

# Print the contents of the Redis list
while(rc.llen('alist')!=0):
    v = rc.lpop('alist')
    print(v, end=" ")
    print(type(v), end="|")
print("")
print( rc.llen('alist') )
print( "-------------------" )
